# Assignment 5 - Evaluation metrics

RAGAS Metrics: https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

## LLM As Judge
1. [Noise sensitivity](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/noise_sensitivity/) - % incorrect claims out of total claims
  - Response
  - Context
2. [Faithfulness](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/faithfulness/) - % of claims supported by retrieved context
  - Response
  - Context
3. Persona preferences - Does the response reflect the preferences of the persona? [0,1,2] scale
  - Response
  - Persona description
  - Gold response example (multi-shot)
  - report as normalized score [0,1]
4. [Answer relevancy](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/answer_relevance/#answer-relevancy) - relevancy of the Gold question to the generated response
  - Gold question
  - Generated response
  - Judge generates 3 questions based on the response
  - Average cosine similarity of generated questions to the gold question


## Mathematical Eval
5. [BERT Score](https://arxiv.org/abs/1904.09675) - compares embeddings of gold asnwer to generates response
  - Gold answer
  - Generated response
6. [Honesty/Hallucination](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/agents/#topic-adherence) - Does the LLM return the "I don't know" phrase [IDK] when it cannot find relevant context (F1 score).
  - Precision = (IDK & NO Context) / (IDK & Have Context + IDK & NO Context)
  - Recall = (IDK & NO Context) / (Factual response & NO Context + IDK Response & NO Context)
  - F1 Score = 2*Precision*Recall / (Precision + Recall)


===========================================================================================================

## 1. Setup

We will first install a number of libraries and import what we will need.





In [1]:
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia

!pip install bert_score

In [3]:
import os
import numpy as np
import time
import locale

# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

locale.getpreferredencoding = lambda: "UTF-8"


/tmp/ipykernel_511/1396652237.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import ArxivLoader


In [4]:
from typing import Annotated, Literal, TypedDict
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers.utils import get_json_schema
from typing import Annotated

In [5]:
%%capture
!pip install -U sentence_transformers
from sentence_transformers import CrossEncoder


!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_50.tar.gz" -C /content

In [ ]:
%%capture

import shutil

class VectorStoreRetriever():
      EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
      def __init__(self):
          self.collection_name = "rag_tech_db_250"
          self.qdrant_path = '/content/qdrant_storage'
          self.init_embeddings(self.EMBEDDINGS_MODEL)
          self.init_vector_store(self.qdrant_path, self.collection_name)
      def init_vector_store(self, path, collection_name):
          self.vector_store = QdrantVectorStore(
              client=QdrantClient(path=self.qdrant_path),
              embedding=self.embeddings,
              collection_name=collection_name,
              distance=Distance.DOT)
      def init_embeddings(self, embeddings_model):
          self.embeddings = HuggingFaceEmbeddings(model_name=embeddings_model)

      def destroy(self):
          self.vector_store.client.delete_collection(self.collection_name)
          if os.path.exists(self.qdrant_path):
              shutil.rmtree(self.qdrant_path)
              print(f"Qdrant storage directory removed ({self.qdrant_path}).")

retriever = VectorStoreRetriever()
vector_store = retriever.vector_store
embedings = retriever.embeddings

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from transformers import GenerationConfig

# LOAD QWEN 3 LLM
# judge for Gold Answer generation
qwen_model_name = "Qwen/Qwen3-8B"

qwen_quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

# load the tokenizer and the model
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float32,
    device_map="auto",
    quantization_config=qwen_quantization_config
)
qwen_model.config.pad_token_id = qwen_model.config.eos_token_id

# Create the generation config object
generation_config = GenerationConfig(
    max_new_tokens=1000,
    temperature=0.3,
    top_p=0.5,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=qwen_model.config.eos_token_id,
    eos_token_id=qwen_model.config.eos_token_id
)

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    generation_config=generation_config
)

qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)
qwen_chat = ChatHuggingFace(llm=qwen_llm)

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate
import re
import json

class ClaimsEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str, context: str):
        content = self.TEMPLATE.format(
            context=context,
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280, do_sample=False)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                response_part = response_part.split("</think>", 1)[1].strip()
        json_output = re.search(r"```(?:json)?\s*\n(.*?)\s*\n```", response_part, re.DOTALL).group(1)
        return json.loads(json_output)

    TEMPLATE = """Your task is to evaluate the claims of a provided statement based ONLY on provided context.

A claim is a single, simplified and separable clause that can be evaluated independently.
You must break the statement into its constituent claims independently of the context.

After the statement is broken into claims, classify each in exactly 1 of 3 possible categories: faithful, noisy, or irrelevant.
A claim is faithful if it is supported or verified by the supplied context. The factual basis of a faithful claim MUST appear in the context.
A claim is noisy if it is incorrect, factually wrong, or contrary to the information in the context.
Claims that are not noisy and not faithful are irrelevant - the context does not mention the information in the claim, so it cannot be determined as noisy or faithful to the context.

Context:
Frogs are green.

Statement:
Darryl is a frog, therefore he is green. Frogs are brown.

Answer:
{{
"claims": [{{ "claim":"Darryl is a frog", "evaluation":"faithful" }},
           {{ "claim": "he is green", "evaluation": "irrelevant" }},
           {{ "claim": "Frogs are brown", "evaluation": "noisy" }}],
"claims_count": 3,
"noisy_count": 1,
"faithful_count": 1,
"irrelevant_count": 1
}}

Context:
{context}

Statement:
{response}

Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"claims": array[{{ "claim":"string","evaluation":string["faithful","noisy","irrelevant"] }}],
"claims_count": integer (total count of all claims),
"noisy_count": integer (count of noisy claims),
"faithful_count": integer (count of faithful claims),
"irrelevant_count": integer (count of irrelevant claims)
}}
```"""

claims_evaluator = ClaimsEvaluator(qwen_tokenizer, qwen_model)

In [ ]:
class PersonaEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str, persona: str):
        content = self.TEMPLATE.format(
            persona=persona,
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280, do_sample=False)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                response_part = response_part.split("</think>", 1)[1].strip()
        json_output = re.search(r"```(?:json)?\s*\n(.*?)\s*\n```", response_part, re.DOTALL).group(1)
        return json.loads(json_output)

    TEMPLATE = """Your task is to evaluate how well a statement reflects the communication preferences of the user.
Some users like detailed and highly technical answers to their questions, while other users like high-level and simplified respones.

Your evaluation must be an integer in [0, 1, 2]
0 - Statement does not reflect the user's preferences
1 - Statement mostly reflects the user's preferences
2 - Statement reflects the user's preferences extremely well

Preferences:
{persona}

Statement:
{response}

Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"persona_reason": string[sentence to justify the score],
"persona_score": integer[0,1,2]
}}
```"""

persona_evaluator = PersonaEvaluator(qwen_tokenizer, qwen_model)

In [ ]:
class RelevancyEvaluator():
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def evaluate(self, response: str):
        content = self.TEMPLATE.format(
            response=response
        )

        message = [
            {"role": "user", "content": content}
        ]

        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280,
                                     top_p=0.95, temperature=1.0)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                response_part = response_part.split("</think>", 1)[1].strip()
        json_output = re.search(r"```(?:json)?\s*\n(.*?)\s*\n```", response_part, re.DOTALL).group(1)
        return json.loads(json_output)

    TEMPLATE = """You are test writing specialist. You will be given a statement that is the intended answer to a question.
Your task is to write a question based ONLY on the content of the statement.
When paired together, the statement should be a reasonable and accurate answer to the question.

Statement:
{response}


Provide 3 DIFFERENT questions. Write ONLY the questions. Do not provide any other text or answer to the question.
Your response MUST be a valid JSON object enclosed in markdown tags structured with this schema:
```json
{{
"questions": ["question 1","question 2","question 3"]
}}
```"""

relevancy_evaluator = RelevancyEvaluator(qwen_tokenizer, qwen_model)


In [ ]:
import bert_score

class BERTScoreEvaluator():

    def evaluate(self, gold_answers: str, responses: str):
        precision, recall, f1 =  bert_score.score(responses, gold_answers, lang="en", verbose=True)
        return {
            "precision": precision.item(),
            "recall": recall.item(),
            "f1": f1.item()
        }

bert_evaluator = BERTScoreEvaluator()


In [ ]:
IDK = "[IDK]"

class HallucinationEvaluator():
    def __init__(self, idk: str):
        self.idk = idk
    def evaluate(self, responses, context):
        has_context = np.array([len(c) > 0 for c in context])
        no_context = np.logical_not(has_context)
        idk = np.array([self.idk in r for r in responses])
        fact = np.logical_not(idk)

        idk_and_context = np.logical_and(idk, has_context)
        idk_and_no_context = np.logical_and(idk, no_context)
        fact_and_context = np.logical_and(fact, has_context)
        fact_and_no_context = np.logical_and(fact, no_context)

        # Precision = (IDK & NO Context) / (IDK & Have Context + IDK & NO Context)
        precision = idk_and_no_context.sum() / (idk_and_context.sum() + idk_and_no_context.sum())
        # Recall = (IDK & NO Context) / (Factual response & NO Context + IDK Response & NO Context)
        recall = idk_and_no_context.sum() / (fact_and_no_context.sum() + idk_and_no_context.sum())
        # F1 Score = 2PrecisionRecall / (Precision + Recall)
        f1 = 2 * precision * recall / (precision + recall)

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

hallucination_evaluator = HallucinationEvaluator()

In [ ]:
class NaiveRAG():
    def __init__(self, tokenizer, model, idk):
        self.tokenizer = tokenizer
        self.model = model
        self.idk = idk

    def invoke(self, question: str, context: str):
        content = self.TEMPLATE.format(
            context=context,
            question=question,
            idk=self.idk
        )
        message = [
            {"role": "user", "content": content}
        ]
        input_ids = self.tokenizer.apply_chat_template(
            message,
            return_tensors="pt",
            enable_thinking=False,
            add_generation_prompt=True
        ).to(self.model.device)

        output = self.model.generate(**input_ids, max_new_tokens=1280,
                                     top_p=0.95, temperature=1.3)
        decoded_output = self.tokenizer.decode(output[0], skip_special_tokens=True)

        return self.extract(decoded_output)

    def extract(self, text_output: str):
        assistant_tag = "assistant"
        if assistant_tag in text_output:
            response_part = text_output.split(assistant_tag, 1)[1].strip()
            if "</think>" in response_part:
                return response_part.split("</think>", 1)[1].strip()
            return response_part

        return text_output.strip()


    TEMPLATE = """You are an expert in generative AI. Your task is to answer the question based ONLY on the provided context.
If the context is blank, irrelevant, unhelpful, useless, or wrong, then simply reply with {idk}.
Do not make up, invent, or return facts from memory. Use only the context to write a response or write {idk}.

Context:
{context}

Question:
{question}

Answer:"""


rag = NaiveRAG(qwen_tokenizer, qwen_model)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

research_persona = """This user is an engineer, who requires detailed technical information when they ask questions.
You will help them by answering about generative AI concepts, internal system architecture, and implementation details."""

marketing_persona = """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
They prefer high level answers that explain concept over technical detail.
You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production."""


test_qs = {"7": {
    "question": "What are some examples of tasks that involve evaluating the robustness of generative AI models?",
    "question_id": 7,
    "context_id": "e196c8cfb0a14c70bb03bb0b370bd0bd",
    "persona": research_persona,
    "metadata": {
      "producer": "pdfTeX-1.40.25",
      "creator": "LaTeX with hyperref",
      "creationdate": "2024-03-28T00:54:45+00:00",
      "source": "https://arxiv.org/pdf/2312.10997.pdf",
      "file_path": "https://arxiv.org/pdf/2312.10997.pdf",
      "total_pages": 21,
      "format": "PDF 1.5",
      "title": "",
      "author": "",
      "subject": "",
      "keywords": "",
      "moddate": "2024-03-28T00:54:45+00:00",
      "trapped": "",
      "modDate": "D:20240328005445Z",
      "creationDate": "D:20240328005445Z",
      "page": 12,
      "page_num": 12,
      "doc_source": "ArXiv",
      "id": "2312.10997",
      "split_id": 68,
      "doc_num": 29
    }
  },
  "8": {
    "question": "How does incorporating task instructions during encoding enhance the versatility of the INSTRUCTOR model for different language tasks?",
    "question_id": 8,
    "context_id": "4f27ed5307dc4aa682291d306299feac",
    "persona": marketing_persona,
    "metadata": {
      "producer": "pdfTeX-1.40.25",
      "creator": "LaTeX with hyperref",
      "creationdate": "2023-05-31T00:45:57+00:00",
      "source": "https://arxiv.org/pdf/2212.09741.pdf",
      "file_path": "https://arxiv.org/pdf/2212.09741.pdf",
      "total_pages": 18,
      "format": "PDF 1.5",
      "title": "",
      "author": "",
      "subject": "",
      "keywords": "",
      "moddate": "2023-05-31T00:45:57+00:00",
      "trapped": "",
      "modDate": "D:20230531004557Z",
      "creationDate": "D:20230531004557Z",
      "page": 1,
      "page_num": 1,
      "doc_source": "ArXiv",
      "id": "2212.09741",
      "split_id": 9,
      "doc_num": 14
    }
  },
  "00": {
    "question": "What is the average flight speed of an unladen swallow?",
    "question_id": 0,
    "context_id": "",
    "persona": marketing_persona
  }
}

responses = []
gold_answers = []
contexts = []

faithfulnesses = []
noisinesses = []
persona_scores = []
relevancy_scores = []
for q in test_qs.values():

    print("*" * 60)
    question = q['question']
    print(f"Question: {question}")
    context_id = q['context_id']

    context = vector_store.get_by_ids([context_id])
    contexts.append(context.content)
    response = rag.invoke(question=question, context=context)
    responses.append(response)
    gold = rag.invoke(question=question, context=context)
    gold_answers.append(gold)
    print("=== CLAIMS ===")
    claims_evaluation = claims_evaluator.evaluate(response=response, context=context)
    for claim in claims_evaluation['claims']:
        print(claim)
    claim_count = claims_evaluation['claims_count']
    faithful_count = claims_evaluation['faithful_count']
    noisy_count = claims_evaluation['noisy_count']
    irrelevant_count = claims_evaluation['irrelevant_count']
    faithful = faithful_count/claim_count
    faithfulnesses.append(faithful)
    noisy = noisy_count/claim_count
    noisinesses.append(noisy)
    print(f"Faithfulness: {(faithful):.1f}")
    print(f"Noisiness: {(noisy):.1f}")
    print()
    print("=== PERSONA ===")
    persona = q['persona']
    persona_evaluation = persona_evaluator.evaluate(response=response, persona=persona)
    persona_score = persona_evaluation['persona_score']
    persona_scores.append(persona_score)
    print(f"Persona thinking: {persona_evaluation['persona_reason']}")
    print(f"Persona score: {persona_evaluation['persona_score']}")
    print()
    print("=== RELEVANCY ===")
    relevancy_evaluation = relevancy_evaluator.evaluate(response=response)
    embeddings = vector_store.embeddings
    embed_question = embeddings.embed_query(question)
    similarity_scores = []
    for n, rel_question in enumerate(relevancy_evaluation['questions']):
        print(f"{n+1}. {rel_question}")
        embed_rel_question = embeddings.embed_query(rel_question)
        similarity_scores.append(cosine_similarity([embed_question], [embed_rel_question])[0][0])
    relevancy_score = np.mean(similarity_scores)
    relevancy_scores.append(relevancy_score)
    print(f"Average cosine similarity: {np.mean(similarity_scores)}")
    print()


print("=" * 60)
print("FINAL EVALUATION")
print("=" * 60)
print()
print("=== LLM-AS-A-JUDGE ===")
print(f"Faithfulness: {np.mean(faithfulnesses):.1f}")
print(f"Noisiness: {np.mean(noisinesses):.1f}")
print(f"Persona: {np.mean(persona_scores):.1f}")
print(f"Relevancy: {np.mean(relevancy_scores):.1f}")
print()
print("=== BERT ===")
bert_evaluation = bert_evaluator.evaluate(gold=gold, response=response)
print(f"Precision: {bert_evaluation['precision']:.3f}")
print(f"Recall: {bert_evaluation['recall']:.3f}")
print(f"F1: {bert_evaluation['f1']:.3f}")
print()
print("=== HALLUCINATION ===")
hallucination_evaluation = hallucination_evaluator.evaluate(responses=responses, context=contexts)
print(f"Precision: {hallucination_evaluation['precision']:.3f}")
print(f"Recall: {hallucination_evaluation['recall']:.3f}")
print(f"F1: {hallucination_evaluation['f1']:.3f}")
print()
